### 통합 실행

In [1]:
# 라이브러리
import sys
import json
import time
from datetime import datetime
from pathlib import Path
from openai import OpenAI
import pandas as pd
import yfinance as yf
import torch

sys.path.append('/Users/honghyeongi/policonomy/third_party/Kronos')
from model import Kronos, KronosTokenizer, KronosPredictor

MODEL = 'gemma4:e4b'
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

Path('../data').mkdir(exist_ok=True)

print(f'시작: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'디바이스: {"mps" if torch.backends.mps.is_available() else "cpu"}')

시작: 2026-04-13 19:09:20
디바이스: mps


In [2]:
# 01_collector 코드 그대로 가져오기
import feedparser
import hashlib

RSS_SOURCES = {
    'politics_economy': [
        'https://www.hankyung.com/feed/economy',
        'https://www.yna.co.kr/rss/economy.xml',
        'https://www.yna.co.kr/rss/politics.xml',
        'https://rss.donga.com/economy.xml',
        'https://rss.donga.com/politics.xml',
        'https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=02',
        'https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=01',
        'https://www.khan.co.kr/rss/rssdata/politic_news.xml',
        'https://www.asiae.co.kr/rss/economy.htm',
    ],
    'global_macro': [
        'https://feeds.feedburner.com/businessinsider',
        'https://rss.nytimes.com/services/xml/rss/nyt/Economy.xml',
        'https://www.investing.com/rss/news.rss',
        'https://www.investing.com/rss/market_overview.rss',
        'https://feeds.marketwatch.com/marketwatch/topstories/',
        'https://www.cnbc.com/id/100003114/device/rss/rss.html',
        'https://www.cnbc.com/id/10000664/device/rss/rss.html',
        'https://www.hankyung.com/feed/international',
        'https://www.yna.co.kr/rss/international.xml',
        'https://www.wired.com/feed/rss',
    ],
    'tech_industry': [
        'https://techcrunch.com/feed/',
        'https://feeds.feedburner.com/venturebeat/SZYF',
        'https://rss.etnews.com/Section901.xml',
        'https://rss.etnews.com/Section902.xml',
        'https://rss.etnews.com/Section903.xml',
        'https://feeds.arstechnica.com/arstechnica/index',
    ],
    'real_estate': [
        'https://www.hankyung.com/feed/realestate',
        'https://www.asiae.co.kr/rss/realestate.htm',
        'https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=07',
        'https://www.mk.co.kr/rss/50200011/',
        'https://www.mk.co.kr/rss/50100032/',
    ],
}

from datetime import timedelta

def collect_rss(category: str, days: int = 7) -> list:
    results = []
    since = datetime.now() - timedelta(days=days)
    for url in RSS_SOURCES.get(category, []):
        try:
            feed = feedparser.parse(url)
            for entry in feed.entries:
                published = entry.get('published_parsed')
                if published:
                    pub_date = datetime(*published[:6])
                    if pub_date < since:
                        continue
                url_hash = hashlib.md5(entry.get('link', '').encode()).hexdigest()
                results.append({
                    'hash': url_hash,
                    'category': category,
                    'title': entry.get('title', ''),
                    'link': entry.get('link', ''),
                    'summary': entry.get('summary', ''),
                    'published': entry.get('published', ''),
                })
        except:
            pass
    return results

def collect_all(days: int = 7) -> dict:
    all_articles = {}
    for category in RSS_SOURCES.keys():
        print(f'  {category} 수집 중...')
        all_articles[category] = collect_rss(category, days)
    return all_articles

print('수집 시작...')
start = time.time()
all_articles = collect_all(days=7)

total = sum(len(v) for v in all_articles.items())
for cat, articles in all_articles.items():
    print(f'  {cat}: {len(articles)}개')
print(f'수집 완료: {time.time()-start:.0f}초')

with open('../data/raw_articles.json', 'w', encoding='utf-8') as f:
    json.dump(all_articles, f, ensure_ascii=False, indent=2)

수집 시작...
  politics_economy 수집 중...
  global_macro 수집 중...
  tech_industry 수집 중...
  real_estate 수집 중...
  politics_economy: 598개
  global_macro: 351개
  tech_industry: 127개
  real_estate: 214개
수집 완료: 7초


In [3]:
# 02_analyzer 코드
from collections import Counter

def deduplicate(articles):
    seen = set()
    unique = []
    for a in articles:
        h = a.get('hash', '')
        if h not in seen:
            seen.add(h)
            unique.append(a)
    return unique

def filter_article(article):
    try:
        response = ollama.chat.completions.create(
            model=MODEL,
            messages=[{'role': 'user', 'content': f'다음 기사가 경제/정치/부동산/테크/금융 중 하나라도 관련있으면 YES, 완전히 관련없으면 NO만 답해.\n제목: {article["title"]}\n답:'}],
            max_tokens=5, temperature=0.1
        )
        return 'YES' in response.choices[0].message.content.strip().upper()
    except:
        return True

def score_article(article):
    try:
        response = ollama.chat.completions.create(
            model=MODEL,
            messages=[{'role': 'user', 'content': f'다음 뉴스를 분석해줘. JSON만 답해.\n제목: {article["title"]}\n요약: {article.get("summary", "")[:200]}\n{{"importance": 1~10, "one_line": "한줄요약", "is_urgent": true또는false, "keywords": ["키워드1"]}}'}],
            max_tokens=200, temperature=0.1
        )
        text = response.choices[0].message.content.strip()
        text = text.replace('```json', '').replace('```', '').strip()
        s = text.find('{')
        e = text.rfind('}') + 1
        if s != -1:
            return json.loads(text[s:e])
    except:
        pass
    return {'importance': 3, 'one_line': article['title'], 'is_urgent': False, 'keywords': []}

def analyze_article(article):
    try:
        response = ollama.chat.completions.create(
            model=MODEL,
            messages=[{'role': 'user', 'content': f'Analyze this Korean news article. Respond in JSON only. No markdown.\n\nTitle: {article["title"]}\nSummary: {article.get("summary", "")[:300]}\nCategory: {article["category"]}\n\n{{"issue": "핵심 이슈 (Korean)", "cause": "원인 (Korean)", "economic_impact": "경제적 파급효과 3문장 (Korean)", "affected_sectors": ["sector1"], "sentiment": "positive", "prediction": "향후 예상 (Korean)", "importance": 5}}'}],
            max_tokens=2048, temperature=0.1
        )
        text = response.choices[0].message.content.strip()
        lines = [l for l in text.split('\n') if not l.strip().startswith('```')]
        text = '\n'.join(lines).strip()
        s = text.find('{')
        e = text.rfind('}') + 1
        if e > 0:
            result = json.loads(text[s:e])
            result['title'] = article['title']
            result['link'] = article.get('link', '')
            result['category'] = article['category']
            return result
    except:
        pass
    return None

# flatten + 중복 제거
articles = []
for category, items in all_articles.items():
    for item in items:
        item['category'] = category
        articles.append(item)
articles = deduplicate(articles)
print(f'중복 제거 후: {len(articles)}개')

# 필터링
print('필터링 중...')
filtered = [a for a in articles if filter_article(a)]
print(f'필터링 후: {len(filtered)}개')

# 스코어링
print('스코어링 중...')
for a in filtered:
    a.update(score_article(a))

# 상위 선별
top_articles = []
for category in RSS_SOURCES.keys():
    cat = sorted([a for a in filtered if a['category'] == category],
                 key=lambda x: x.get('importance', 0), reverse=True)
    top_articles.extend(cat)

urgent = [a for a in filtered if a.get('is_urgent') and a not in top_articles]
top_articles.extend(urgent)
print(f'선별 완료: {len(top_articles)}개')

# 심층 분석
print('심층 분석 중...')
analyzed = []
for i, article in enumerate(top_articles):
    print(f'  {i+1}/{len(top_articles)} - {article["title"][:40]}')
    result = analyze_article(article)
    if result:
        analyzed.append(result)
    time.sleep(0.1)

with open('../data/analyzed_articles.json', 'w', encoding='utf-8') as f:
    json.dump(analyzed, f, ensure_ascii=False, indent=2)
print(f'분석 완료: {len(analyzed)}개')

중복 제거 후: 1264개
필터링 중...
필터링 후: 48개
스코어링 중...
선별 완료: 48개
심층 분석 중...
  1/48 - 반도체 호황發 '세수 풍년'…법인·소득·거래세 다 늘었다
  2/48 - 코오롱그룹, 첨단소재·바이오·수소탱크…신산업 투자 확대
  3/48 - 노동부, 산재 고위험사업장 10만곳 전수조사…법위반시 강력조치
  4/48 - 농식품·중기부, 팁스 농식품기업 16곳에 30억∼60억원씩 지원
  5/48 - 4월 1~10일 수출 36.7% 늘어 ‘역대 최고’…반도체 152.5% 
  6/48 - 한은 “가상자산 거래소에도 서킷 브레이커 도입 필요”
  7/48 - 이 대통령 “한-폴란드 ‘포괄적 전략 동반자 관계’ 격상…방산 협력 의미
  8/48 - 靑 “경제 불확실성 여전…종전 선언까지 비상대응체제 유지”
  9/48 - "자동 주문(API)으로 가상자산 거래 부풀려 시세 조종"…금감원, 기획
  10/48 - 반도체 수출 호조…1일~10일 수출액 또 역대 최대
  11/48 - 대외경제협력기금 연 3조 원 목표…AI·문화·공급망 사업 지원
  12/48 - 올해 가계대출 바짝 조인다…5대은행 증가율 1% 안팎서 묶일 듯
  13/48 - 1분기 전기차 구매 20대가 가장 많아…보조금·가격 인하 통했다
  14/48 - 송언석 "주거비 부담 서민·청년에게 전가…서울 공급 확대해야"
  15/48 - “한국은 미국 다음 동맹국” 한·폴란드 정상회담, 포괄적 전략 동반자 관
  16/48 - 당정 “차량 5·2부제 따른 자동차 보험료 인하 방안 다음주 발표”
  17/48 - 금융위, '특금법 위반' 코인원에 영업일부정지 3개월·과태료 52억원 제
  18/48 - 지자체·공공기관 등 150여곳에 성평등 조직문화 조성 설명회
  19/48 - 불공정거래 악용 쉬운 가상자산 API…금감원, 투자자 주의 당부
  20/48 - 4월 1~10일 수출 252억달러 '역대 최대'…또 반도체가 끌었다 (상
  21/48 - [속보] 4월 1~

In [4]:
# 03_predictor 코드
import numpy as np

tokenizer = KronosTokenizer.from_pretrained('NeoQuasar/Kronos-Tokenizer-base')
model = Kronos.from_pretrained('NeoQuasar/Kronos-small')
predictor = KronosPredictor(model, tokenizer, max_context=512)

def get_ohlcv(ticker, period='6mo'):
    stock = yf.Ticker(ticker)
    df = stock.history(period=period)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
    df.columns = ['open', 'high', 'low', 'close', 'volume']
    df.index = pd.to_datetime(df.index)
    return df.dropna()

def predict_market(name, df, pred_days=5):
    lookback = min(400, len(df) - pred_days)
    x_df = df.iloc[:lookback][['open', 'high', 'low', 'close', 'volume']].reset_index(drop=True)
    x_ts = pd.Series(pd.to_datetime(df.index[:lookback]).tz_localize(None) if df.index.tz is not None else df.index[:lookback])
    y_ts = pd.Series(pd.to_datetime(df.index[lookback:lookback+pred_days]).tz_localize(None) if df.index.tz is not None else df.index[lookback:lookback+pred_days])
    try:
        pred_df = predictor.predict(df=x_df, x_timestamp=x_ts, y_timestamp=y_ts, pred_len=pred_days, T=1.0, top_p=0.9, sample_count=3)
        last_close = float(df['close'].iloc[lookback-1])
        pred_close = float(pred_df['close'].iloc[-1])
        change_pct = (pred_close - last_close) / last_close * 100
        return {
            'name': name,
            'last_close': round(last_close, 2),
            'pred_close': round(pred_close, 2),
            'change_pct': round(change_pct, 2),
            'direction': '상승' if change_pct > 0 else '하락',
            'pred_range': {'high': round(float(pred_df['high'].max()), 2), 'low': round(float(pred_df['low'].min()), 2)}
        }
    except Exception as e:
        print(f'{name} 예측 실패: {e}')
        return None

def fuse_predictions(predictions, analyzed_articles):
    top_issues = [a['issue'] for a in analyzed_articles[:5]]
    issues_text = '\n'.join(f'- {issue}' for issue in top_issues)
    kronos_summary = '\n'.join([f'- {v["name"]}: {v["direction"]} ({v["change_pct"]:+.2f}%)' for v in predictions.values() if v])
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': f'Respond in JSON only. No markdown.\n\nMarket predictions:\n{kronos_summary}\n\nTop news:\n{issues_text}\n\n{{"kospi": {{"direction": "하락", "reason": "reason", "confidence": 70}}, "usd_krw": {{"direction": "약세", "reason": "reason", "confidence": 65}}, "key_sector": {{"name": "반도체", "direction": "강세", "reason": "reason"}}, "overall_sentiment": "negative", "weekly_summary": "summary"}}'}],
        max_tokens=2048, temperature=0.1
    )
    text = response.choices[0].message.content.strip()
    lines = [l for l in text.split('\n') if not l.strip().startswith('```')]
    text = '\n'.join(lines).strip()
    s = text.find('{')
    e = text.rfind('}') + 1
    if e > 0:
        try:
            return json.loads(text[s:e])
        except:
            pass
    return {}

targets = {'kospi': '^KS11', 'usd_krw': 'KRW=X', 'samsung': '005930.KS', 'nasdaq': '^IXIC'}
market_data = {name: get_ohlcv(ticker) for name, ticker in targets.items()}

predictions = {}
for name, df in market_data.items():
    result = predict_market(name, df)
    if result:
        predictions[name] = result
        print(f'{name}: {result["direction"]} ({result["change_pct"]:+.2f}%)')

fused = fuse_predictions(predictions, analyzed)

final_prediction = {
    'generated_at': datetime.now().isoformat(),
    'kronos_predictions': predictions,
    'fused_analysis': fused
}
with open('../data/predictions.json', 'w', encoding='utf-8') as f:
    json.dump(final_prediction, f, ensure_ascii=False, indent=2)
print('예측 완료')

100%|██████████| 5/5 [15:03<00:00, 180.66s/it]


kospi: 하락 (-1.48%)


100%|██████████| 5/5 [00:00<00:00,  7.14it/s]


usd_krw: 하락 (-0.26%)


100%|██████████| 5/5 [00:00<00:00, 20.40it/s]


samsung: 상승 (+1.26%)


100%|██████████| 5/5 [00:00<00:00, 31.45it/s]


nasdaq: 하락 (-0.01%)
예측 완료


In [5]:
# 04_newsletter 코드
from jinja2 import Template

TEMPLATE = open('../templates/newsletter.html', 'r', encoding='utf-8').read()

def build_newsletter_data(analyzed, predictions):
    categories = {
        'politics_economy': '정치/경제',
        'global_macro': '글로벌 매크로',
        'tech_industry': '산업/테크',
        'real_estate': '부동산',
    }
    sections = []
    for cat_key, cat_name in categories.items():
        articles = sorted([a for a in analyzed if a.get('category') == cat_key],
                          key=lambda x: x.get('importance', 0), reverse=True)
        if articles:
            sections.append({'category': cat_name, 'articles': articles})

    fused = predictions.get('fused_analysis', {})
    return {
        'date': datetime.now().strftime('%Y년 %m월 %d일'),
        'sections': sections,
        'urgent': [a for a in analyzed if a.get('importance', 0) >= 8],
        'prediction': fused,
        'kronos': predictions.get('kronos_predictions', {}),
        'weekly_summary': fused.get('weekly_summary', ''),
    }

data = build_newsletter_data(analyzed, final_prediction)
html = Template(TEMPLATE).render(**data)

with open('../data/newsletter.html', 'w', encoding='utf-8') as f:
    f.write(html)
print('뉴스레터 생성 완료')

뉴스레터 생성 완료


In [6]:
import subprocess

elapsed = time.time() - start
print(f'\n전체 파이프라인 완료!')
print(f'소요 시간: {elapsed/60:.1f}분')
print(f'수집:   {sum(len(v) for v in all_articles.values())}개')
print(f'분석:   {len(analyzed)}개')
print(f'예측:   {len(predictions)}개 지수')
print(f'뉴스레터: data/newsletter.html')

subprocess.run(['open', '../data/newsletter.html'])


전체 파이프라인 완료!
소요 시간: 521.7분
수집:   1290개
분석:   48개
예측:   4개 지수
뉴스레터: data/newsletter.html


CompletedProcess(args=['open', '../data/newsletter.html'], returncode=0)

In [2]:
import re
text = '''Thinking...
Thinking Process:
...done thinking.

{"importance": 5, "one_line": "중국 3월 수출 증가세 둔화", "is_urgent": false}'''

text = re.sub(r'Thinking\.\.\.[\s\S]*?\.\.\.done thinking\.', '', text)
print(repr(text))

'\n\n{"importance": 5, "one_line": "중국 3월 수출 증가세 둔화", "is_urgent": false}'
